In [7]:
import pandas as pd
import numpy as np
from datetime import datetime

RAW = r'D:\Dev\20. Prueba_Tecnica-Compartamos\data_raw'

# RAW = todo string (así lo deja la ingesta)
# Lectura de datos

df_customers = pd.read_csv(f'{RAW}\\customers.csv', dtype=str)
df_products  = pd.read_csv(f'{RAW}\\products.csv',  dtype=str)
df_orders    = pd.read_csv(f'{RAW}\\orders.csv',    dtype=str)

print('customers', df_customers.shape)
print('products ', df_products.shape)
print('orders   ', df_orders.shape)

customers (109, 10)
products  (47, 8)
orders    (520, 12)


In [8]:
# Vistazo general: nulos y duplicados por tabla
for nombre, df in [('customers', df_customers), ('products', df_products), ('orders', df_orders)]:
    print('=' * 45)
    print(f'  {nombre.upper()}   filas={len(df)}  cols={df.shape[1]}')
    print('=' * 45)
    print('Nulos por columna:')
    print(df.isnull().sum().to_string())
    print(f'Filas 100% nulas : {df.isnull().all(axis=1).sum()}')
    print(f'Filas duplicadas : {df.duplicated().sum()}')
    print()

  CUSTOMERS   filas=109  cols=10
Nulos por columna:
customer_id           3
first_name            6
last_name             3
email                 3
phone                 8
city                  3
country               3
age                  31
registration_date     3
loyalty_tier         13
Filas 100% nulas : 3
Filas duplicadas : 8

  PRODUCTS   filas=47  cols=8
Nulos por columna:
product_id       0
product_name     0
category         0
price_usd        0
cost_usd         3
stock_units     24
supplier        12
active           0
Filas 100% nulas : 0
Filas duplicadas : 0

  ORDERS   filas=520  cols=12
Nulos por columna:
order_id               5
customer_id            5
product_id             5
quantity               5
unit_price_usd         5
total_amount_usd       5
order_date             5
ship_date              5
status                 5
payment_method         5
discount_pct         111
credit_card_last4      5
Filas 100% nulas : 5
Filas duplicadas : 19



In [9]:
# Helper de fechas: ya que hay fechas que estan en formatos mixtos
def parse_fecha(serie):
    s = serie.str.strip().str.replace(r'[./]', '-', regex=True)
    return pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')



In [10]:
#LIMPIEZA DE CLIENTES

# Evidencia de problemas
vc = df_customers['customer_id'].value_counts()
print('customer_id duplicados :', vc[vc > 1].to_dict())
print('customer_id nulos      :', df_customers['customer_id'].isna().sum())
print('country (variantes)    :', sorted(df_customers['country'].dropna().unique()))
print('loyalty_tier (variantes):', sorted(df_customers['loyalty_tier'].dropna().unique()), '| nulos:', df_customers['loyalty_tier'].isna().sum())
print('age (rango)            :', sorted(pd.to_numeric(df_customers['age'], errors='coerce').dropna().unique()))
print('registration_date (muestras):', df_customers['registration_date'].dropna().unique()[:6].tolist())

customer_id duplicados : {'1078': 2, '1026': 2, '1068': 2, '1054': 2, '1065': 2, '1021': 2}
customer_id nulos      : 3
country (variantes)    : ['CO', 'COLOMBIA', 'Col.', 'Colombia', 'colombia']
loyalty_tier (variantes): ['Bronze', 'GOLD', 'Gold', 'Silver', 'bronze'] | nulos: 13
age (rango)            : [np.float64(-5.0), np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(24.0), np.float64(25.0), np.float64(26.0), np.float64(27.0), np.float64(29.0), np.float64(33.0), np.float64(39.0), np.float64(43.0), np.float64(44.0), np.float64(46.0), np.float64(47.0), np.float64(54.0), np.float64(60.0), np.float64(65.0), np.float64(200.0)]
registration_date (muestras): ['2024-02-22', '2023.09.05', '2024/03/12', '2023/05/15', '2024/03/01', '2024-01-13']


In [11]:
#Se detecta duplicados, tambien hay nulos en id, distinta varianes en country, loyalty_tier, age con valores no numéricos y fechas con formatos mixtos.

In [12]:
df = df_customers.copy()

# 1) Filas idénticas y filas 100% nulas
df = df.drop_duplicates().dropna(how='all')

# 2) customer_id: sin ID no sirve el registro -> drop, y a entero
df = df.dropna(subset=['customer_id'])
df['customer_id'] = pd.to_numeric(df['customer_id'], errors='coerce').astype('Int64')

# 3) registration_date a DATE (la necesito ANTES del dedup para elegir el más antiguo)
df['registration_date'] = parse_fecha(df['registration_date'])

# 4) Dedup por PK: en duplicados conservo la fecha de registro MÁS ANTIGUA (indicación).
#    Ordeno ascendente por fecha (nulos al final) y me quedo con el primero.
df = (df.sort_values('registration_date', na_position='last')
        .drop_duplicates(subset=['customer_id'], keep='first'))

# 5) Texto a MAYÚSCULA
for col in ['first_name', 'last_name', 'city']:
    df[col] = df[col].str.upper().str.strip()

# 6) País único -> COLOMBIA (todas las variantes apuntan al mismo país)
df['country'] = 'COLOMBIA'

# 7) age: edades imposibles (<0 o >120) -> NULL
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df.loc[(df['age'] < 0) | (df['age'] > 120), 'age'] = np.nan
df['age'] = df['age'].astype('Int64')

# 8) loyalty_tier: válidos {BRONZE, SILVER, GOLD}; inválido/nulo -> NO DETERMINADO
TIERS = {'BRONZE', 'SILVER', 'GOLD'}
df['loyalty_tier'] = df['loyalty_tier'].str.upper().str.strip()
df['loyalty_tier'] = df['loyalty_tier'].where(df['loyalty_tier'].isin(TIERS), 'NO DETERMINADO')

# 9) Auditoría
df['stage_updated_at'] = pd.Timestamp.now()

df_customers_clean = df.copy()
print(f'customers: {len(df_customers)} -> {len(df_customers_clean)} filas')

customers: 109 -> 100 filas


In [13]:
# Verificación
print('PK duplicados   :', df_customers_clean['customer_id'].duplicated().sum())
print('country         :', df_customers_clean['country'].unique().tolist())
print('loyalty_tier    :', df_customers_clean['loyalty_tier'].unique().tolist())
print('age min/max     :', df_customers_clean['age'].min(), '/', df_customers_clean['age'].max())
print('reg_date nulos  :', df_customers_clean['registration_date'].isna().sum())
df_customers_clean.head()

PK duplicados   : 0
country         : ['COLOMBIA']
loyalty_tier    : ['GOLD', 'SILVER', 'BRONZE', 'NO DETERMINADO']
age min/max     : 20 / 65
reg_date nulos  : 0


,customer_id,first_name,last_name,email,phone,city,country,age,registration_date,loyalty_tier,stage_updated_at
48,1008,MARÍA,VARGAS,maria.vargas9@gmail.com,+57 3249877065,BOGOTA,COLOMBIA,<NA>,2023-01-12,GOLD,2026-06-17 09:55:22.778646
9,1015,MATEO,DÍAZ,mateo.diaz32@gmail.com,+57 3128434500,MEDELLIN,COLOMBIA,<NA>,2023-01-16,GOLD,2026-06-17 09:55:22.778646
51,1039,LAURA,CASTRO,laura.castro9@gmail.com,+57 3111520596,BARRANQUILLA,COLOMBIA,<NA>,2023-01-21,SILVER,2026-06-17 09:55:22.778646
102,1071,MATEO,VARGAS,mateo.vargas49@gmail.com,+57 3212817745,BARRANQUILLA,COLOMBIA,<NA>,2023-01-26,GOLD,2026-06-17 09:55:22.778646
97,1037,JUAN,MORALES,juan.morales22@gmail.com,+57 3149961107,BOGOTA,COLOMBIA,<NA>,2023-01-28,BRONZE,2026-06-17 09:55:22.778646


In [14]:
#LIMPIAMOS PRODUCTOS

# Evidencia
vc = df_products['product_id'].value_counts()
print('product_id duplicados:', vc[vc > 1].to_dict())
print('price_usd <= 0       :', df_products[pd.to_numeric(df_products['price_usd'], errors='coerce') <= 0]['price_usd'].tolist())
print('cost_usd nulos       :', df_products['cost_usd'].isna().sum())
print('stock_units nulos    :', df_products['stock_units'].isna().sum())
print('supplier (variantes) :', sorted(df_products['supplier'].dropna().unique()), '| nulos:', df_products['supplier'].isna().sum())
print('active (variantes)   :', sorted(df_products['active'].dropna().unique()))
print('category (variantes) :', sorted(df_products['category'].dropna().unique()))

product_id duplicados: {'2036': 2, '2019': 2, '2018': 2, '2023': 2}
price_usd <= 0       : ['0', '-1424.71', '0']
cost_usd nulos       : 3
stock_units nulos    : 24
supplier (variantes) : ['Proveedor C', 'ProveedorA', 'ProveedorB', 'proveedor_a'] | nulos: 12
active (variantes)   : ['0', '1', 'FALSE', 'True', 'false', 'true']
category (variantes) : ['Beauty', 'Books', 'Clothing', 'Electronics', 'Home & Kitchen', 'Sports', 'Toys', 'electronics']


In [15]:
# Hay duplicados, precios negativos o cero, 
# nulos en cost y stock, distintas variantes en supplier, active y category.

In [ ]:
#comenzamos limpieza de productos

df = df_products.copy()

# 1) Duplicados / filas nulas
df = df.drop_duplicates().dropna(how='all')

# 2) product_id a entero + dedup por PK (eliminar 1)
df['product_id'] = pd.to_numeric(df['product_id'], errors='coerce').astype('Int64')
df = df.drop_duplicates(subset=['product_id'], keep='first')

# 3) Texto a MAYÚSCULA
for col in ['product_name', 'category']:
    df[col] = df[col].str.upper().str.strip()

# 4) price/cost/stock: inválido (nulo o <= 0) -> 0
for col, es_entero in [('price_usd', False), ('cost_usd', False), ('stock_units', True)]:
    v = pd.to_numeric(df[col], errors='coerce')
    v = v.where(v > 0, 0)
    df[col] = v.astype('Int64') if es_entero else v.round(2)

# 5) supplier -> formato 'PROVEEDOR [LETRA]'; sin letra reconocible -> NO DETERMINADO
letra = df['supplier'].str.upper().str.extract(r'([A-Z])\s*$')[0]
df['supplier'] = ('PROVEEDOR ' + letra).where(letra.notna(), 'NO DETERMINADO')

# 6) active -> booleano (solo 2 valores)
TRUTHY = {'1', 'TRUE', 'T', 'YES', 'SI'}
df['active'] = df['active'].str.upper().str.strip().isin(TRUTHY)

# 7) Auditoría
df['stage_updated_at'] = pd.Timestamp.now()

df_products_clean = df.copy()
print(f'products: {len(df_products)} -> {len(df_products_clean)} filas')

products: 47 -> 43 filas


In [17]:
# Verificación
print('PK duplicados :', df_products_clean['product_id'].duplicated().sum())
print('supplier      :', sorted(df_products_clean['supplier'].unique()))
print('active         :', df_products_clean['active'].unique().tolist())
print('price<0/costNaN/stockNaN:', (df_products_clean['price_usd'] < 0).sum(), df_products_clean['cost_usd'].isna().sum(), df_products_clean['stock_units'].isna().sum())
df_products_clean.head()

PK duplicados : 0
supplier      : ['NO DETERMINADO', 'PROVEEDOR A', 'PROVEEDOR B', 'PROVEEDOR C']
active         : [False, True]
price<0/costNaN/stockNaN: 0 0 0


,product_id,product_name,category,price_usd,cost_usd,stock_units,supplier,active,stage_updated_at
0,2043,JUEGO DE MESA,TOYS,841.55,579.97,0,PROVEEDOR C,False,2026-06-17 09:55:22.844841
1,2038,BASE MAQUILLAJE,BEAUTY,264.06,144.68,348,PROVEEDOR A,False,2026-06-17 09:55:22.844841
2,2014,SUDADERA HOODIE,CLOTHING,192.73,92.67,0,PROVEEDOR C,False,2026-06-17 09:55:22.844841
3,2015,PANTALON CARGO,CLOTHING,106.76,70.09,474,PROVEEDOR C,False,2026-06-17 09:55:22.844841
4,2013,ZAPATOS RUNNING,CLOTHING,586.61,237.41,0,PROVEEDOR A,True,2026-06-17 09:55:22.844841


In [18]:
#LIMPIAMOS ORDENES

# Evidencia
vc = df_orders['order_id'].value_counts()
print('order_id duplicados :', (vc > 1).sum(), 'PKs')
print('filas 100% nulas    :', df_orders.isnull().all(axis=1).sum())
print('quantity <= 0       :', (pd.to_numeric(df_orders['quantity'], errors='coerce') <= 0).sum())
print('status (variantes)  :', sorted(df_orders['status'].dropna().unique()))
print('payment (variantes) :', sorted(df_orders['payment_method'].dropna().unique()))
print('discount_pct (rango):', sorted(pd.to_numeric(df_orders['discount_pct'], errors='coerce').dropna().unique()))
print('order_date (muestras):', df_orders['order_date'].dropna().unique()[:6].tolist())

order_id duplicados : 15 PKs
filas 100% nulas    : 5
quantity <= 0       : 52
status (variantes)  : ['COMPLETED', 'Cancelled', 'Completed', 'Pending', 'cancelled', 'completed', 'pending', 'returned']
payment (variantes) : ['Credit Card', 'PayPal', 'cash', 'credit_card', 'debit_card', 'paypal']
discount_pct (rango): [np.float64(0.0), np.float64(5.0), np.float64(10.0), np.float64(15.0), np.float64(20.0)]
order_date (muestras): ['2023/11/15', '2023/09/15', '2023/03/03', '2024/04/06', '2023/11/26', '2024/01/17']


In [19]:
# Hay duplicados, filas 100% nulas, 
# quantity con valores no numéricos ,
# distintas variantes en status y payment_method, 
# discount_pct con valores no numéricos o fuera de rango, fechas con formatos mixtos.

In [ ]:
#empezamos limpieza de ordenes

df = df_orders.copy()

# 1) Duplicados / filas nulas
df = df.drop_duplicates().dropna(how='all')

# 2) order_id no nulo + identificadores a entero
df = df.dropna(subset=['order_id'])
for col in ['order_id', 'customer_id', 'product_id']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# 3) Dedup por PK
df = df.drop_duplicates(subset=['order_id'], keep='first')

# 4) Montos/descuento a numérico
for col in ['unit_price_usd', 'total_amount_usd', 'discount_pct']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 5) quantity inválida (nula o <= 0) -> recalcular total/unit_price
#    (verifiqué que total = quantity * unit_price, el descuento no está incluido)
q = pd.to_numeric(df['quantity'], errors='coerce')
mask_q = q.isna() | (q <= 0)
recalc = (df['total_amount_usd'] / df['unit_price_usd']).round()
df['quantity'] = q.where(~mask_q, recalc).astype('Int64')

# 6) Fechas a DATE; ship_date < order_date no tiene sentido -> NULL
df['order_date'] = parse_fecha(df['order_date'])
df['ship_date']  = parse_fecha(df['ship_date'])
df.loc[df['ship_date'] < df['order_date'], 'ship_date'] = pd.NaT

# 7) status estandarizado; inválido -> NO DETERMINADO
STATUS = {'COMPLETED', 'PENDING', 'CANCELLED', 'RETURNED'}
df['status'] = df['status'].str.upper().str.strip()
df['status'] = df['status'].where(df['status'].isin(STATUS), 'NO DETERMINADO')

# 8) payment_method estandarizado (espacios -> _); inválido -> NO DETERMINADO
PAY = {'CREDIT_CARD', 'DEBIT_CARD', 'PAYPAL', 'CASH'}
df['payment_method'] = df['payment_method'].str.upper().str.strip().str.replace(' ', '_')
df['payment_method'] = df['payment_method'].where(df['payment_method'].isin(PAY), 'NO DETERMINADO')


In [ ]:
# Al ser una tabla intermedio tenemos que verificar integridad referencial con clientes y productos.
# cliente y producto deben existir en sus tablas

ids_c = set(df_customers_clean['customer_id'].dropna())
ids_p = set(df_products_clean['product_id'].dropna())

sin_cliente  = ~df['customer_id'].isin(ids_c)
sin_producto = ~df['product_id'].isin(ids_p)
es_huerfana  = sin_cliente | sin_producto

# Detalle: cuáles órdenes son huérfanas y por qué
huerfanas = df.loc[es_huerfana, ['order_id', 'customer_id', 'product_id']].copy()
huerfanas['motivo'] = np.select(
    [sin_cliente[es_huerfana] & sin_producto[es_huerfana], sin_cliente[es_huerfana]],
    ['CLIENTE Y PRODUCTO INEXISTENTES', 'CLIENTE INEXISTENTE'],
    default='PRODUCTO INEXISTENTE')

print(f'orders huérfanas: {len(huerfanas)}  '
      f'(sin cliente={int(sin_cliente.sum())}, sin producto={int(sin_producto.sum())})')
print(huerfanas['motivo'].value_counts().to_string())
print()
print(huerfanas.to_string(index=False))

df = df[~es_huerfana]

# 10) Auditoría
df['stage_updated_at'] = pd.Timestamp.now()

df_orders_clean = df.copy()
print(f'\norders: {len(df_orders)} -> {len(df_orders_clean)} filas')

orders huérfanas: 27  (sin cliente=12, sin producto=15)
motivo
PRODUCTO INEXISTENTE    15
CLIENTE INEXISTENTE     12

 order_id  customer_id  product_id               motivo
     3186         1075        8888 PRODUCTO INEXISTENTE
     3416         1065        8888 PRODUCTO INEXISTENTE
     3225         1028        8888 PRODUCTO INEXISTENTE
     3089         1007        8888 PRODUCTO INEXISTENTE
     3412         9999        2039  CLIENTE INEXISTENTE
     3002         1070        8888 PRODUCTO INEXISTENTE
     3231         1028        8888 PRODUCTO INEXISTENTE
     3152         1007        8888 PRODUCTO INEXISTENTE
     3283         1058        8888 PRODUCTO INEXISTENTE
     3397         1048        8888 PRODUCTO INEXISTENTE
     3309         1007        8888 PRODUCTO INEXISTENTE
     3487         9999        2038  CLIENTE INEXISTENTE
     3379         9999        2033  CLIENTE INEXISTENTE
     3072         1028        8888 PRODUCTO INEXISTENTE
     3345         1003        8888 PRODUCT

In [22]:
# Verificamos si todo ta bien
print('PK duplicados      :', df_orders_clean['order_id'].duplicated().sum())
print('quantity <= 0      :', (df_orders_clean['quantity'] <= 0).sum())
print('status             :', sorted(df_orders_clean['status'].unique()))
print('payment_method     :', sorted(df_orders_clean['payment_method'].unique()))
print('ship_date < order  :', (df_orders_clean['ship_date'] < df_orders_clean['order_date']).sum())
df_orders_clean.head()

PK duplicados      : 0
quantity <= 0      : 0
status             : ['CANCELLED', 'COMPLETED', 'PENDING', 'RETURNED']
payment_method     : ['CASH', 'CREDIT_CARD', 'DEBIT_CARD', 'PAYPAL']
ship_date < order  : 0


,order_id,customer_id,product_id,quantity,unit_price_usd,total_amount_usd,order_date,ship_date,status,payment_method,discount_pct,credit_card_last4,stage_updated_at
0,3149,1064,2018,9,1385.04,12465.36,2023-11-15,2023-11-20,COMPLETED,PAYPAL,5.0,8454,2026-06-17 09:55:22.932804
1,3286,1048,2035,9,85.43,768.87,2023-09-15,2023-09-28,PENDING,CASH,NaN,7025,2026-06-17 09:55:22.932804
2,3451,1046,2004,3,52.98,158.94,2023-03-03,2023-03-17,CANCELLED,CREDIT_CARD,10.0,4602,2026-06-17 09:55:22.932804
3,3300,1024,2019,4,480.09,1920.36,2024-04-06,2024-04-11,CANCELLED,DEBIT_CARD,15.0,3762,2026-06-17 09:55:22.932804
4,3355,1014,2006,5,487.29,2436.45,2023-11-26,2023-12-07,COMPLETED,CREDIT_CARD,0.0,4310,2026-06-17 09:55:22.932804


In [23]:
#chequeo de integridad referencial
ids_c = set(df_customers_clean['customer_id'])
ids_p = set(df_products_clean['product_id'])
print('orders sin customer válido:', (~df_orders_clean['customer_id'].isin(ids_c)).sum())
print('orders sin product válido :', (~df_orders_clean['product_id'].isin(ids_p)).sum())

orders sin customer válido: 0
orders sin product válido : 0
